# Scaling: distortion plots, physics scaling plots

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys


sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
device = "cuda"

In [ ]:
from typing import Sequence, Callable, Optional, Dict

import re
from pathlib import Path
from functools import partial
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import torch

from neugk.dataset import CycloneAEDataset
from neugk.pinc.neural_fields.trad import (
    zfp_recon,
    wavelet_recon,
    pca_recon,
    jpeg2000_recon,
)
from neugk.pinc.neural_fields.data import CycloneNFDataset
from neugk.pinc.neural_fields.nf_utils import sample_field, load_nf, compress_weights

from neugk.integrals import FluxIntegral
from neugk.pinc.autoencoders.ae_utils import load_autoencoder

In [ ]:
TIMESTEPS = [100, 140, 200]
TRAJECTORIES = [
    "iteration_13.h5",
    "iteration_115.h5",
    "iteration_131.h5",
    "iteration_134.h5",
    "iteration_146.h5",
    "iteration_148.h5",
    "iteration_160.h5",
    "iteration_200.h5",
    "iteration_210.h5",
    "iteration_212.h5",
]

## Load all models

### Neural fields

In [ ]:
TAG = ""

MODEL = "mlp"
ckp_dir = Path("../nf_ckps_scale")

nfs = {}
nf_has_int = set()

pattern = re.compile(rf"{TAG}{MODEL}_([\w.]+)_t(\d+)_x(\d+)\.pt$")
for file in ckp_dir.glob(f"{TAG}{MODEL}_*.pt"):
    m = pattern.match(file.name)
    if not m:
        continue
    traj, t, cr = m.groups()
    traj = traj.split(".")[0]
    t, cr = int(t), int(cr)

    if cr not in nfs:
        nfs[cr] = {}
    if traj not in nfs[cr]:
        nfs[cr][traj] = {}
    if t not in nfs[cr][traj]:
        nfs[cr][traj][t] = {}

    nfs[cr][traj][t] = load_nf(file.as_posix(), device)

# integrals (overwrite if existing)
pattern = re.compile(rf"{TAG}int_{MODEL}_([\w.]+)_t(\d+)_x(\d+)\.pt$")
for file in ckp_dir.glob(f"{TAG}int_{MODEL}_*.pt"):
    m = pattern.match(file.name)
    if not m:
        continue
    traj, t, cr = m.groups()
    traj = traj.split(".")[0]
    t, cr = int(t), int(cr)

    if cr not in nfs:
        nfs[cr] = {}
    if traj not in nfs[cr]:
        nfs[cr][traj] = {}
    if t not in nfs[cr][traj]:
        nfs[cr][traj][t] = {}

    nfs[cr][traj][t] = load_nf(file.as_posix(), device)
    nf_has_int.add(cr)
nfs.keys()

### Autoencoders

In [ ]:
ae_root = "/system/user/publicwork/gyrokinetics_checkpoints/compression_ckpts"

In [ ]:
# AEs
aes = []
cr_302 = f"{ae_root}/20250911_144548/best.pth"
ae, _, ae_cfg = load_autoencoder(cr_302, device=device)
aes.append(ae)
cr_1208 = f"{ae_root}/20250911_143423/best.pth"
ae, _, ae_cfg = load_autoencoder(cr_1208, device=device)
aes.append(ae)
# # NOTE: this model has issues in the evaluation
# cr_2865 = f"{ae_root}/20250917_133131/best.pth"
# ae, _, ae_cfg = load_autoencoder(cr_2865, device=device)
# aes.append(ae)
cr_4835 = f"{ae_root}/20250911_143546/best.pth"
ae, _, ae_cfg = load_autoencoder(cr_4835, device=device)
aes.append(ae)
cr_19342 = f"{ae_root}/20250916_095521/best.pth"
ae, _, ae_cfg = load_autoencoder(cr_19342, device=device)
aes.append(ae)

# # AEs + PEFT
# # NOTE: we take the ckp and not the best, as the best is from first epoch (worse on pinn losses)
# aes_peft = []
# cr_1208_peft = f"{ae_root}/20250920_190044/ckp.pth"
# ae_peft, _, ae_cfg = load_autoencoder(cr_1208_peft, device=device, load_peft=True)
# aes_peft.append(ae_peft)

# VQVAEs
vqvaes = []
cr_19342 = f"{ae_root}/20250911_143024/best.pth"
vqvae, _, vqvae_cfg = load_autoencoder(cr_19342, device=device)
vqvaes.append(vqvae)
cr_25789 = f"{ae_root}/20250919_204102/best.pth"
vqvae, _, vqvae_cfg = load_autoencoder(cr_25789, device=device)
vqvaes.append(vqvae)
# NOTE: this model has issues in the evaluation
# cr_38684 = f"{ae_root}/20250911_143106/best.pth"
# vqvae, _, vqvae_cfg = load_autoencoder(cr_38684, device=device)
# vqvaes.append(vqvae)
cr_77368 = f"{ae_root}/20250911_143815/best.pth"
vqvae, _, vqvae_cfg = load_autoencoder(cr_77368, device=device)
vqvaes.append(vqvae)

# # VQVAEs + PEFT
# vqvaes_peft = []
# cr_77368_peft = f"{ae_root}/20250919_073133/best.pth"
# vqvae_peft, _, vqvae_cfg = load_autoencoder(cr_77368_peft, device=device, load_peft=True)
# vqvaes_peft.append(vqvae_peft)

## Utils

In [ ]:
def ml_eval(
    pred_df: torch.Tensor,
    gt_df: torch.Tensor,
    pred_phi: torch.Tensor,
    gt_phi: torch.Tensor,
    pred_eflux: torch.Tensor,
    gt_eflux: torch.Tensor,
    compressed_size: int = None,
):
    metrics = {}

    metrics["mse"] = ((pred_df.cpu() - gt_df.cpu()) ** 2).mean()
    metrics["l1"] = (pred_df.cpu() - gt_df.cpu()).abs().mean()
    metrics["psnr"] = 10 * torch.log10(gt_df.max() ** 2 / metrics["mse"])
    metrics["phi_mse"] = ((pred_phi - gt_phi) ** 2).mean()
    metrics["phi_l1"] = (pred_phi - gt_phi).abs().mean()
    metrics["phi_psnr"] = 10 * torch.log10(gt_phi.max() ** 2 / metrics["phi_mse"])
    metrics["eflux_l1"] = (pred_eflux.sum() - gt_eflux.sum()).abs().mean()
    metrics["eflux_field_l1"] = (pred_eflux - gt_eflux).abs().mean()
    metrics["eflux_field_mse"] = ((pred_eflux - gt_eflux) ** 2).mean()
    metrics["eflux_field_psnr"] = 10 * torch.log10(
        gt_eflux.max() ** 2 / metrics["eflux_field_mse"]
    )

    if compressed_size is not None:
        num_pixels = pred_df.numel()
        metrics["bpp"] = (compressed_size * 8) / num_pixels

    return metrics

In [ ]:
@torch.no_grad()
def run_eval_diagnostics(
    trajectories: str,
    timesteps: Sequence[int],
    models: Sequence[Callable],
    model_name: str,
    device: torch.device,
    cfg: Optional[Dict] = None,
):
    ml_metrics_at_cr = {}
    # gt_dfs_dict = {}
    for model in tqdm(models, desc=model_name, total=len(models)):
        ml_metrics = defaultdict(list)
        for traj in trajectories:
            data = CycloneNFDataset(
                traj, timesteps=timesteps, normalize=None, realpotens=True
            )
            gt_dfs = [data.df[:, t] for t in range(data.df.shape[1])]
            # gt_dfs_dict[traj] = gt_dfs
            compressed_size = None

            if "NF" in model_name:
                # nf reconstruct
                dfs = []
                compressed_size = 0
                for t in range(data.df.shape[1]):
                    # TODO load config
                    real_t = timesteps[t]
                    traj = traj.split(".")[0]
                    if "ZFP" in model_name:

                        nf = model[traj][real_t].to(device)
                        # nf_full_size = sum(p.nbytes for p in nf.parameters())
                        nf, _, nf_zfp_size = compress_weights(nf, tolerance=1e-3)
                        # print(f"NF + ZFP extra: {(nf_full_size / nf_zfp_size):.2f}x")
                        compressed_size += nf_zfp_size
                    else:
                        nf = model[traj][real_t].to(device)
                        compressed_size += sum(p.nbytes for p in nf.parameters())
                    # create new normalized dataset
                    nf_data = CycloneNFDataset(
                        traj,
                        timesteps=real_t,
                        normalize="zscore",
                        normalize_coords="discrete" not in nf.embed_type,
                        separate_ky_modes=None,
                        realpotens=True,
                    )
                    # automatically denormalizes
                    dfs.append(sample_field(nf, nf_data, device).cpu())

                    torch.cuda.empty_cache()

            if "AE" in model_name or "VQ-VAE" in model_name:
                dfs = []
                compressed_size = 0

                for t in range(data.df.shape[1]):
                    model = model.to(device)
                    # bit triky here, need the original config and datasets (normalization)
                    traindata_ae = CycloneAEDataset(
                        path=cfg.dataset.path,
                        split="train",
                        input_fields=["df", "phi", "flux"],
                        random_seed=cfg.seed,
                        normalization=cfg.dataset.normalization,
                        normalization_scope=cfg.dataset.normalization_scope,
                        trajectories=cfg.dataset.training_trajectories,
                        separate_zf=cfg.dataset.separate_zf,
                        real_potens=True,
                        cond_filters=vars(cfg.dataset.training_cond_filters),
                        stage="autoencoder",
                        conditions=["itg", "dg", "s_hat", "q"],
                    )
                    valdata_ae = CycloneAEDataset(
                        path=cfg.dataset.path,
                        split="train",
                        input_fields=["df", "phi", "flux"],
                        random_seed=cfg.seed,
                        normalization=cfg.dataset.normalization,
                        normalization_scope=cfg.dataset.normalization_scope,
                        normalization_stats=traindata_ae.norm_stats,
                        trajectories=[traj],
                        separate_zf=cfg.dataset.separate_zf,
                        real_potens=True,
                        stage="autoencoder",
                        conditions=["itg", "dg", "s_hat", "q"],
                    )
                    # ae reconstruct
                    sample = valdata_ae[timesteps[t]]
                    df = sample.df.unsqueeze(0).to(device)
                    condition = sample.conditioning.unsqueeze(0).to(device)
                    ae_df = model(df, condition=condition)["df"].cpu().squeeze(0)
                    # important: denormalize
                    ae_df = valdata_ae.denormalize(0, ae_df)
                    if ae_df.shape[0] == 4:
                        ae_df = ae_df[[0, 1]] + ae_df[[2, 3]]
                    dfs.append(ae_df)

                    torch.cuda.empty_cache()

                    if "VQ-VAE" in model_name:
                        model_output = model(df, condition=condition)
                        # get indicies (for VQ-VAE only they are saved)
                        indices = model_output["vq_indices"]

                        # indices to int16 for higher compression
                        # (codebook_size=8192 fits in int16)
                        indices_int16 = indices.to(torch.int16)
                        compressed_size += indices_int16.nbytes
                    else:
                        # AE
                        latent = model.encode(df, condition=condition)[0]
                        compressed_size += latent.nbytes

            if model_name in ["ZFP", "Wavelet", "PCA", "JPEG2000"]:
                dfs = []
                compressed_size = 0
                for t in range(data.df.shape[1]):
                    recon, _, cs = model(data.full_df[:, t])
                    dfs.append(recon)
                    compressed_size += cs
            if compressed_size:
                ml_metrics["cr"].append(data.full_df.nbytes / compressed_size)

            geom = {k: v.unsqueeze(0) for k, v in data.geom.items()}
            # ml metrics
            integral = FluxIntegral(flux_fields=True)
            if model_name != "GT":
                for pred_df, gt_df in zip(dfs, gt_dfs):
                    pred_phi, (_, pred_eflux, _) = integral(geom, pred_df.unsqueeze(0))
                    gt_phi, (_, gt_eflux, _) = integral(geom, gt_df.unsqueeze(0))

                    ml_eval_dict = ml_eval(
                        pred_df,
                        gt_df,
                        pred_phi,
                        gt_phi,
                        pred_eflux,
                        gt_eflux,
                        compressed_size,
                    )
                    for k, v in ml_eval_dict.items():
                        ml_metrics[k].append(v)
        ml_metrics = {k: float(sum(v) / len(v)) for k, v in ml_metrics.items()}
        ml_metrics_at_cr[int(ml_metrics["cr"])] = ml_metrics
        # if "df" not in ml_metrics_at_cr[int(ml_metrics["cr"])]:
        #     ml_metrics_at_cr[int(ml_metrics["cr"])]["df"] = {}
        # ml_metrics_at_cr[int(ml_metrics["cr"])]["df"][traj] = dfs
    return {model_name: ml_metrics_at_cr}

## Evaluation

In [ ]:
full_metrics = {}

In [ ]:
# NF
nf_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=list(nfs.values()),
    model_name="NF",
    device=device,
)
full_metrics.update(nf_metrics)

# NF + ZFP
nf_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=list(nfs.values()),
    model_name="NF + ZFP",
    device=device,
)
full_metrics.update(nf_metrics)

In [ ]:
# AE
ae_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=aes,
    model_name="AE",
    cfg=ae_cfg,
    device=device,
)
full_metrics.update(ae_metrics)

# VQ-VAE
vqvae_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=vqvaes,
    model_name="VQ-VAE",
    cfg=vqvae_cfg,
    device=device,
)
full_metrics.update(vqvae_metrics)

# # AE + PEFT
# ae_metrics = run_eval_diagnostics(
#     TRAJECTORIES,
#     timesteps=TIMESTEPS,
#     models=aes_peft,
#     model_name="AE + PEFT",
#     cfg=ae_cfg,
#     device=device,
# )
# full_metrics.update(ae_metrics)

# # VQ-VAE + PEFT
# vqvae_metrics = run_eval_diagnostics(
#     TRAJECTORIES,
#     timesteps=TIMESTEPS,
#     models=vqvaes_peft,
#     model_name="VQ-VAE + PEFT",
#     cfg=vqvae_cfg,
#     device=device,
# )
# full_metrics.update(vqvae_metrics)

In [ ]:
# ZFP
zfp_recons = [
    partial(zfp_recon, tolerance=tol) for tol in [20, 50, 100, 500, 1000, 2500, 5000]
]
zfp_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=zfp_recons,
    model_name="ZFP",
    device=device,
)
full_metrics.update(zfp_metrics)

# wavelet
wavelet_recons = [
    partial(wavelet_recon, threshold=tol) for tol in [1, 5, 10, 25, 50, 100]
]
wft_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=wavelet_recons,
    model_name="Wavelet",
    device=device,
)
full_metrics.update(wft_metrics)

# PCA
pca_recons = [partial(pca_recon, n_components=n) for n in [1, 2, 4, 8, 16, 32]]
pca_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=pca_recons,
    model_name="PCA",
    device=device,
)
full_metrics.update(pca_metrics)

# JPEG2000
jp2k_recons = [partial(jpeg2000_recon, quality=q) for q in [0.001, 0.1, 1.0, 5.0, 10.0]]
jp2k_metrics = run_eval_diagnostics(
    TRAJECTORIES,
    timesteps=TIMESTEPS,
    models=jp2k_recons,
    model_name="JPEG2000",
    device=device,
)
full_metrics.update(jp2k_metrics)

## Compression scaling

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

methods = list(full_metrics.keys())
for m in ["GT", "AE + PEFT", "VQ-VAE + PEFT"]:
    if m in methods:
        methods.remove(m)

cmap = plt.cm.plasma_r
colors = [np.array([0, 0, 0, 1])] + list(
    cmap(np.linspace(0.1, 0.8, len(methods) - 2 - 1))
)

fig, ax = plt.subplots(figsize=(7, 5))

group_colors = {}
group_linestyles = {}
color_iter = iter(colors)

for method in methods:
    if "NF" in method:
        if "NF" not in group_colors:
            group_colors["NF"] = next(color_iter)
        color = group_colors["NF"]
        linestyle = "-" if method == "NF" else "--"
        group_linestyles[method] = linestyle
    elif "AE" in method:
        if "AE" not in group_colors:
            group_colors["AE"] = next(color_iter)
        color = group_colors["AE"]
        linestyle = "-" if method == "AE" else "--"
        group_linestyles[method] = linestyle
    else:
        color = next(color_iter)
        linestyle = "-"
        group_colors[method] = color
        group_linestyles[method] = linestyle

    crs = sorted(full_metrics[method].keys())
    psnrs = [full_metrics[method][cr]["psnr"] for cr in crs]

    if method == "AE":
        crs, psnrs = crs[:-1], psnrs[:-1]

    zorder = 1
    lw = 3.5
    if "AE" in method or "NF" in method:
        zorder = 2
        lw = 4

    ax.plot(
        crs,
        psnrs,
        marker="o",
        label=method,
        color=color,
        lw=lw,
        markersize=10,
        linestyle=linestyle,
        markeredgecolor="white",
        markeredgewidth=0.0,
        zorder=zorder,
    )

ax.set_xscale("log")
ax.set_xlabel("CR (log)", fontsize=18)
ax.set_ylabel("PSNR [dB]", fontsize=18)
ax.tick_params(axis="both", which="major", labelsize=14)
ax.tick_params(axis="both", which="minor", labelsize=12)
ax.grid(True, which="both", ls="--", alpha=0.2)

ax.legend(
    fontsize=13,
    title_fontsize=13,
    frameon=True,
    fancybox=True,
    shadow=False,
    ncols=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.20),
)
fig.tight_layout()
fig.savefig("scaling_psnr.pdf", bbox_inches="tight", dpi=300)

## Physics loss scaling

In [ ]:
methods = list(full_metrics.keys())
for m in ["GT", "NF + ZFP", "AE", "VQ-VAE", "AE + PEFT", "VQ-VAE + PEFT"]:
    if m in methods:
        methods.remove(m)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

for method in methods:
    color = group_colors[method]
    crs = sorted(full_metrics[method].keys())
    fluxl1, phil1 = [], []
    crs_real = []

    for cr in crs:
        if "NF" not in method or cr in nf_has_int:
            crs_real.append(cr)
            fluxl1.append(full_metrics[method][cr]["eflux_l1"])
            phil1.append(full_metrics[method][cr]["phi_l1"])

    axes[0].plot(
        crs_real,
        fluxl1,
        marker="o",
        label=method,
        color=color,
        lw=4,
        markersize=8,
        linestyle=linestyle,
        markeredgecolor="white",
        markeredgewidth=0.0,
    )
    axes[1].plot(
        crs_real,
        phil1,
        marker="o",
        label=method,
        color=color,
        lw=4,
        markersize=8,
        linestyle=linestyle,
        markeredgecolor="white",
        markeredgewidth=0.0,
    )

for ax in axes:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(True, which="both", ls="--", alpha=0.2)

ae_cr = list(full_metrics["VQ-VAE + PEFT"])[-1]

# Scatter points
y_vqvae = full_metrics["VQ-VAE"][ae_cr]["eflux_l1"]
y_peft = full_metrics["VQ-VAE + PEFT"][ae_cr]["eflux_l1"]
y2_vqvae = full_metrics["VQ-VAE"][ae_cr]["phi_l1"]
y2_peft = full_metrics["VQ-VAE + PEFT"][ae_cr]["phi_l1"]

axes[0].scatter(
    ae_cr, y_vqvae, marker="x", label="VQ-VAE", color="#025396", s=100, zorder=102, lw=4
)
axes[0].scatter(
    ae_cr, y_peft, marker="o", label="VQ-VAE + EVA", color="#0BCC0B", s=100, zorder=100
)
axes[1].scatter(
    ae_cr,
    y2_vqvae,
    marker="x",
    label="VQ-VAE",
    color="#025396",
    s=100,
    zorder=102,
    lw=4,
)
axes[1].scatter(
    ae_cr, y2_peft, marker="o", label="VQ-VAE + EVA", color="#0BCC0B", s=100, zorder=100
)

# Draw arrow showing improvement
axes[0].annotate(
    "",
    xy=(ae_cr, y_peft),
    xytext=(ae_cr, y_vqvae),
    arrowprops=dict(facecolor="black", arrowstyle="->", lw=2),
    zorder=101,
)
axes[1].annotate(
    "",
    xy=(ae_cr, y2_peft),
    xytext=(ae_cr, y2_vqvae),
    arrowprops=dict(facecolor="black", arrowstyle="->", lw=2),
    zorder=101,
)

axes[1].set_xlabel("CR (log)", fontsize=16)
axes[0].set_ylabel(r"$\mathcal{L}_Q$", fontsize=16)
axes[1].set_ylabel(r"$\mathcal{L}_{\phi}$", fontsize=16)

axes[1].set_ylim(top=2e3)
axes[0].tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

axes[0].legend(
    fontsize=12,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.40),
    ncol=4,
    # frameon=False
)

fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.01)
fig.savefig("scaling_pinns.pdf", bbox_inches="tight", dpi=300)

## Qualitative df plot

In [ ]:
TRAJ = "iteration_212"
T = 140

data = CycloneNFDataset(
    TRAJ,
    timesteps=T,
    normalize="zscore",
    normalize_coords=False,
    separate_ky_modes=None,
    realpotens=True,
)

with torch.no_grad():
    xnf = sample_field(
        nfs[1163]["iteration_212"][T].to(device), data.to(device), device
    ).cpu()
xwft, _, _ = wavelet_recon(data.full_df.cpu(), threshold=10)
xgt = data.full_df.cpu()

dfs = [xgt, xnf, xgt - xnf, xwft, xwft - xgt]

In [ ]:
def plot_fields_grid(
    x,
    projections,
    space=5,
    suptitle=None,
    titles=None,
    row_labels=None,
    cmap="RdBu_r",
    use_labels=False,
    use_colorbar=True,
):
    if space == 5:
        all_labels = [r"$v_{\parallel}$", r"$\mu$", r"$s$", r"$x$", r"$y$"]
    else:
        all_labels = [r"$x$", r"$s$", r"$y$"]

    if isinstance(x, list):
        x = torch.stack(x, 0)

    num_cols = x.shape[0]
    num_rows = len(projections)

    fig, ax = plt.subplots(num_rows, num_cols, figsize=(2 * num_cols, 0.94 * num_rows))
    fig.subplots_adjust(
        left=0.06, right=0.95, top=0.95, bottom=0, wspace=0.0, hspace=0.0
    )
    if num_rows == 1:
        ax = ax[None, :]
    if num_cols == 1:
        ax = ax[:, None]

    ims = []
    for i, proj in enumerate(projections):
        data_list = [x[j].mean(0).sum(proj) for j in range(num_cols)]
        vmin, vmax = float(min(d.min() for d in data_list)), float(
            max(d.max() for d in data_list)
        )
        vmid = 0.5 * (vmin + vmax)

        row_ims = []
        for j in range(num_cols):
            im = ax[i, j].matshow(
                data_list[j], cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto"
            )
            row_ims.append(im)
            ax[i, j].set_xticks([])
            ax[i, j].set_yticks([])
            if i == 0 and titles is not None:
                ax[i, j].set_title(
                    rf"$t={titles[j]:.1f}R/V_{{\mathrm{{r}}}}$", fontsize=18
                )
        ims.append(row_ims)

        if row_labels is not None:
            # Fix row label alignment - center it vertically for the row
            ax[i, 0].set_ylabel(row_labels[i], fontsize=16)
            # Adjust label position to be centered
            ax[i, 0].yaxis.set_label_coords(-0.15, 0.5)  # Adjust x-coordinate as needed

        if use_labels:
            label_text = "/".join(
                [all_labels[o] for o in range(space) if o not in proj]
            )
            # Fix vertical positioning of labels
            fig.text(
                0.02,
                1.0 - (i + 0.5) / num_rows,  # Center vertically in each row
                label_text,
                va="center",
                ha="center",
                fontsize=18,
            )

        if use_colorbar:
            # Get the position of the entire row (use first subplot in row as reference)
            pos = ax[i, 0].get_position()
            # Calculate the vertical center of the row
            row_center_y = pos.y0 + pos.height / 2

            # Create colorbar with fixed height centered on the row
            cax_height = pos.height * 0.8  # 70% of row height
            cax_y0 = row_center_y - cax_height / 2  # Center vertically

            cax = fig.add_axes([0.97, cax_y0, 0.01, cax_height])
            cbar = fig.colorbar(row_ims[0], cax=cax)
            cbar.set_ticks([0.8 * vmin, 0.8 * vmax])
            cbar.ax.yaxis.set_major_formatter(
                plt.FuncFormatter(lambda val, _: f"{val:.1e}")
            )
            cbar.ax.tick_params(labelsize=10)

    if suptitle is not None:
        fig.text(1.05, 0.5, suptitle, va="center", ha="left", fontsize=20, rotation=-90)
    fig.patch.set_alpha(0.0)
    return fig

In [ ]:
fig = plot_fields_grid(
    dfs,
    use_labels=True,
    projections=[(0, 1, 2), (2, 3, 4), (1, 3, 4), (0, 1, 3), (0, 1, 4)],
)
fig.savefig("df_recon2.pdf", bbox_inches="tight")

## Qualitative phi plot

In [ ]:
with torch.no_grad():
    xnf = sample_field(
        nfs[1163]["iteration_212"][T].to(device), data.to(device), device
    ).cpu()
xwft, _, _ = wavelet_recon(data.full_df.cpu(), threshold=10)
xgt = data.full_df.cpu()

geom = {k: v.unsqueeze(0).cpu() for k, v in data.geom.items()}
integral = FluxIntegral(flux_fields=True)
phinf = integral(geom, xnf.unsqueeze(0))[0].squeeze(0)
phiwft = integral(geom, xwft.unsqueeze(0))[0].squeeze(0)
phigt = integral(geom, xgt.unsqueeze(0))[0].squeeze(0)
phinf = phinf / phinf.max()
phiwft = phiwft / phiwft.max()
phigt = phigt / phigt.max()

phis = [phigt, phinf, phigt - phinf, phiwft, phiwft - phigt]

In [ ]:
fig = plot_fields_grid(
    phis, use_labels=True, projections=[(0,), (1,), (2,)], space=3, cmap="plasma"
)
fig.savefig("phi_recon2.pdf", bbox_inches="tight")

## Appendix plots

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

methods = list(full_metrics.keys())
# methods.remove("PCA")

cmap = plt.cm.plasma_r
colors = [np.array([0, 0, 0, 1])] + list(
    cmap(np.linspace(0.1, 0.8, len(methods) - 2 - 1))
)

fig, ax = plt.subplots(figsize=(14, 10))

group_colors = {}
group_linestyles = {}
color_iter = iter(colors)

for method in methods:
    if "NF" in method:
        if "NF" not in group_colors:
            group_colors["NF"] = next(color_iter)
        color = group_colors["NF"]
        linestyle = "-" if method == "NF" else "--"
        group_linestyles[method] = linestyle
    elif "AE" in method:
        if "AE" not in group_colors:
            group_colors["AE"] = next(color_iter)
        color = group_colors["AE"]
        linestyle = "-" if method == "AE" else "--"
        group_linestyles[method] = linestyle
    else:
        color = next(color_iter)
        linestyle = "-"
        group_colors[method] = color
        group_linestyles[method] = linestyle

    crs = sorted(full_metrics[method].keys())
    psnrs = [full_metrics[method][cr]["psnr"] for cr in crs]

    if method == "AE":
        crs, psnrs = crs[:-1], psnrs[:-1]

    zorder = 0
    lw = 4
    if "AE" in method or "NF" in method:
        zorder = 1
        lw = 5

    if "PEFT" not in method:
        ax.plot(
            crs,
            psnrs,
            marker="o",
            label=method,
            color=color,
            lw=lw,
            markersize=10,
            linestyle=linestyle,
            markeredgecolor="white",
            markeredgewidth=0.0,
            zorder=zorder,
        )
    else:
        ax.scatter(
            crs,
            psnrs,
            marker="X",
            label=method.replace("PEFT", "EVA"),
            color="#0BCC0B" if "VQ-VAE" in method else "#315831",
            lw=1,
            s=200,
        )

ax.set_xscale("log")
ax.set_xlabel("CR (log)", fontsize=18)
ax.set_ylabel("PSNR [dB]", fontsize=18)
ax.tick_params(axis="both", which="major", labelsize=14)
ax.tick_params(axis="both", which="minor", labelsize=12)
ax.grid(True, which="both", ls="--", alpha=0.2)

ax.legend(fontsize=13, title_fontsize=13, frameon=True, fancybox=True, shadow=False)
fig.tight_layout()
fig.savefig("scaling_psnr_app.pdf", bbox_inches="tight", dpi=300)

In [ ]:
methods = list(full_metrics.keys())
for m in ["GT"]:
    # for m in ["GT", "NF + ZFP", "AE", "VQ-VAE"]:
    if m in methods:
        methods.remove(m)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for method in methods:
    if "AE" in method:
        color = group_colors["AE"]
        linestyle = "--"
    elif "+ ZFP" in method:
        color = group_colors["NF"]
        linestyle = "--"
    else:
        color = group_colors[method]
        linestyle = group_linestyles[method]
    crs = sorted(full_metrics[method].keys())
    fluxl1, phil1 = [], []
    crs_real = []

    for cr in crs:
        if method == "NF + ZFP":
            if cr > 1500:
                crs_real.append(cr)
                fluxl1.append(full_metrics[method][cr]["eflux_l1"])
                phil1.append(full_metrics[method][cr]["phi_l1"])
        elif method != "NF" or cr in nf_has_int:
            crs_real.append(cr)
            fluxl1.append(full_metrics[method][cr]["eflux_l1"])
            phil1.append(full_metrics[method][cr]["phi_l1"])

    if "PEFT" not in method:
        axes[0].plot(
            crs_real,
            fluxl1,
            marker="o",
            label=method,
            color=color,
            lw=4.5,
            markersize=8,
            linestyle=linestyle,
            markeredgecolor="white",
            markeredgewidth=0.0,
        )
        axes[1].plot(
            crs_real,
            phil1,
            marker="o",
            label=method,
            color=color,
            lw=4.5,
            markersize=8,
            linestyle=linestyle,
            markeredgecolor="white",
            markeredgewidth=0.0,
        )

    else:
        axes[0].scatter(
            crs,
            fluxl1,
            marker="X",
            label=method.replace("PEFT", "EVA"),
            color="#0BCC0B" if "VQ-VAE" in method else "#315831",
            lw=1,
            s=200,
            zorder=100,
        )
        axes[1].scatter(
            crs,
            phil1,
            marker="X",
            label=method.replace("PEFT", "EVA"),
            color="#0BCC0B" if "VQ-VAE" in method else "#315831",
            lw=1,
            s=200,
            zorder=100,
        )

for ax in axes:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(True, which="both", ls="--", alpha=0.2)

axes[1].set_xlabel("CR (log)", fontsize=16)
axes[0].set_ylabel(r"$\mathcal{L}_Q$", fontsize=16)
axes[1].set_ylabel(r"$\mathcal{L}_{\phi}$", fontsize=16)

axes[1].set_ylim(top=2e3)

axes[0].tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

axes[0].legend(title="Method", fontsize=12, title_fontsize=13, frameon=False)

fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.01)

fig.savefig("scaling_pinn_app.pdf", bbox_inches="tight", dpi=300)

## Extra df plots

In [ ]:
TRAJECTORIES = [
    "iteration_13",
    "iteration_115",
    "iteration_131",
    "iteration_134",
    "iteration_146",
    "iteration_148",
    "iteration_160",
    "iteration_200",
    "iteration_210",
    "iteration_212",
]

T = 140

dfs_extras = []
phis_extras = []
for i, traj in enumerate(TRAJECTORIES):
    data = CycloneNFDataset(
        traj,
        timesteps=T,
        normalize="zscore",
        normalize_coords=False,
        separate_ky_modes=None,
        realpotens=True,
    )

    with torch.no_grad():
        xnf = sample_field(nfs[1163][traj][T].to(device), data.to(device), device).cpu()
    xwft, _, _ = wavelet_recon(data.full_df.cpu(), threshold=10)
    xgt = data.full_df.cpu()

    dfs = [xgt, xnf, xgt - xnf, xwft, xwft - xgt]
    dfs_extras.append(dfs)

    geom = {k: v.unsqueeze(0).cpu() for k, v in data.geom.items()}
    integral = FluxIntegral(flux_fields=True)
    phinf = integral(geom, xnf.unsqueeze(0))[0].squeeze(0)
    phiwft = integral(geom, xwft.unsqueeze(0))[0].squeeze(0)
    phigt = integral(geom, xgt.unsqueeze(0))[0].squeeze(0)
    phinf = phinf / phinf.max()
    phiwft = phiwft / phiwft.max()
    phigt = phigt / phigt.max()

    phis = [phigt, phinf, phigt - phinf, phiwft, phiwft - phigt]
    phis_extras.append(phis)

    fig = plot_fields_grid(
        dfs,
        use_labels=True,
        projections=[(0, 1, 2), (2, 3, 4), (1, 3, 4), (0, 1, 3), (0, 1, 4)],
    )
    fig.savefig(f"figs/extra_df_recon{i}.pdf", bbox_inches="tight")

    fig = plot_fields_grid(
        phis, use_labels=True, projections=[(0,), (1,), (2,)], space=3, cmap="plasma"
    )
    fig.savefig(f"figs/extra_phi_recon{i}.pdf", bbox_inches="tight")